<a href="https://colab.research.google.com/github/chilakagopichand-dotcom/AI-Resume-Analyzer-and-Job-Recommendation-System/blob/main/AI_Resume_Analyzer_and_Job_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive and Save Files
Run the cell below to connect your Google Drive to your Colab session. This allows you to read and write files directly from/to your Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Create a target folder inside your Google Drive if it doesn't exist
import os
target_folder = '/content/drive/MyDrive/ResumeProject'
os.makedirs(target_folder, exist_ok=True)
print(f"Google Drive is mounted. Target folder ready at: {target_folder}")

Mounted at /content/drive
Google Drive is mounted. Target folder ready at: /content/drive/MyDrive/ResumeProject


### Copy 'Resume.csv' to Google Drive
If you have already uploaded `Resume.csv` to the temporary Colab storage, run the cell below to copy it permanently to your Google Drive.

In [2]:
import shutil

source_file = 'Resume.csv'
destination_path = '/content/drive/MyDrive/ResumeProject/Resume.csv'

if os.path.exists(source_file):
    shutil.copy(source_file, destination_path)
    print(f"Successfully copied {source_file} to Google Drive at: {destination_path}")
else:
    print(f"Error: {source_file} was not found in your current Colab directory.")
    print("Please upload it to Colab first using the sidebar files tab, then run this cell.")

Error: Resume.csv was not found in your current Colab directory.
Please upload it to Colab first using the sidebar files tab, then run this cell.


In [ ]:
!pip install pandas numpy scikit-learn
!pip install nltk PyMuPDF
!pip install matplotlib seaborn
!pip install flask

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
import os

print("Uploaded files:", os.listdir())

Uploaded files: ['.config', 'drive', 'sample_data']


In [4]:
import pandas as pd
import os

file_path = "Resume.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    print("Dataset shape:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
else:
    print(f"Error: The file '{file_path}' was not found in the current directory.")
    print("Please upload 'Resume.csv' using the left sidebar in Colab before running this cell.")

Dataset loaded successfully!
Dataset shape: (2484, 4)

Column names:
['ID', 'Resume_str', 'Resume_html', 'Category']

First 5 rows:


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [5]:
print("Dataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nNumber of Resume Categories:")
print(df["Category"].nunique())

print("\nResume Category Distribution:")
print(df["Category"].value_counts())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB
None

Missing Values:
ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

Number of Resume Categories:
24

Resume Category Distribution:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTI

In [3]:

from google.colab import files

uploaded = files.upload()

Saving naukri_com-job_sample.csv to naukri_com-job_sample.csv
Saving resume_model.pkl to resume_model.pkl
Saving Resume.csv to Resume.csv
Saving tfidf_vectorizer.pkl to tfidf_vectorizer.pkl


In [6]:
# Select only the required columns
df_clean = df[["Resume_str", "Category"]].copy()

# Remove rows with missing values
df_clean = df_clean.dropna(subset=["Resume_str", "Category"])

# Convert values into strings
df_clean["Resume_str"] = df_clean["Resume_str"].astype(str)
df_clean["Category"] = df_clean["Category"].astype(str)

# Remove extra spaces
df_clean["Resume_str"] = df_clean["Resume_str"].str.strip()
df_clean["Category"] = df_clean["Category"].str.strip()

# Remove empty records
df_clean = df_clean[
    (df_clean["Resume_str"] != "") &
    (df_clean["Category"] != "")
]

# Remove duplicate records
df_clean = df_clean.drop_duplicates(
    subset=["Resume_str", "Category"]
)

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("Data cleaning completed successfully!")
print("Original dataset records:", len(df))
print("Clean dataset records:", len(df_clean))

print("\nCleaned dataset:")
display(df_clean.head())

Data cleaning completed successfully!
Original dataset records: 2484
Clean dataset records: 2481

Cleaned dataset:


,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADM...,HR
1,"HR SPECIALIST, US HR OPERATIONS Summary ...",HR
2,HR DIRECTOR Summary Over 20 years e...,HR
3,"HR SPECIALIST Summary Dedicated, Driv...",HR
4,HR MANAGER Skill Highlights ...,HR


In [7]:
import re

# Function to clean resume text
def clean_text(text):

    text = text.lower()

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Keep letters, numbers, +, # and .
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# Apply preprocessing
df_clean["Cleaned_Resume"] = df_clean["Resume_str"].apply(clean_text)

print("Resume text preprocessing completed!")

print("\nOriginal Resume:")
print(df_clean["Resume_str"].iloc[0][:300])

print("\nCleaned Resume:")
print(df_clean["Cleaned_Resume"].iloc[0][:300])

Resume text preprocessing completed!

Original Resume:
HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to c

Cleaned Resume:
hr administrator marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Input features and target labels
X = df_clean["Cleaned_Resume"]
y = df_clean["Category"]

# Split dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

# Convert text into numerical features
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF feature extraction completed!")

print("\nTraining records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("Training feature shape:", X_train_tfidf.shape)
print("Testing feature shape:", X_test_tfidf.shape)

TF-IDF feature extraction completed!

Training records: 1984
Testing records: 497
Training feature shape: (1984, 5000)
Testing feature shape: (497, 5000)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)

# Train the model
lr_model.fit(X_train_tfidf, y_train)

# Predict resume categories
y_pred_lr = lr_model.predict(X_test_tfidf)

# Calculate accuracy
lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Model Training Completed!")

print("\nModel Accuracy:", round(lr_accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression Model Training Completed!

Model Accuracy: 65.39 %

Classification Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.33      0.54      0.41        24
           AGRICULTURE       1.00      0.46      0.63        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.85      0.74      0.79        23
               BANKING       0.80      0.70      0.74        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.55      0.92      0.69        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.77      0.77      0.77        22
            CONSULTANT       0.50      0.22      0.30        23
        

In [10]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Create Linear SVM model
svm_model = LinearSVC()

# Train the model
svm_model.fit(X_train_tfidf, y_train)

# Predict resume categories
y_pred_svm = svm_model.predict(X_test_tfidf)

# Calculate accuracy
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Model Training Completed!")

print("\nLinear SVM Accuracy:",
      round(svm_accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    zero_division=0
))

Linear SVM Model Training Completed!

Linear SVM Accuracy: 73.24 %

Classification Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.61      0.83      0.70        24
              ADVOCATE       0.71      0.71      0.71        24
           AGRICULTURE       0.80      0.62      0.70        13
               APPAREL       0.57      0.42      0.48        19
                  ARTS       0.62      0.38      0.47        21
            AUTOMOBILE       1.00      0.43      0.60         7
              AVIATION       0.94      0.74      0.83        23
               BANKING       0.85      0.74      0.79        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.71      0.83      0.77        24
                  CHEF       0.86      0.79      0.83        24
          CONSTRUCTION       0.79      0.86      0.83        22
            CONSULTANT       0.81      0.57      0.67        23
            

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train_tfidf, y_train)

# Predict resume categories
y_pred_rf = rf_model.predict(X_test_tfidf)

# Calculate accuracy
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Model Training Completed!")

print("\nRandom Forest Accuracy:",
      round(rf_accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_rf,
    zero_division=0
))

Random Forest Model Training Completed!

Random Forest Accuracy: 72.64 %

Classification Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.55      0.96      0.70        24
              ADVOCATE       0.83      0.79      0.81        24
           AGRICULTURE       1.00      0.38      0.56        13
               APPAREL       0.86      0.32      0.46        19
                  ARTS       0.60      0.14      0.23        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.83      0.87      0.85        23
               BANKING       0.80      0.70      0.74        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.75      0.88      0.81        24
                  CHEF       0.79      0.79      0.79        24
          CONSTRUCTION       0.83      0.91      0.87        22
            CONSULTANT       0.71      0.43      0.54        23
      

In [12]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Random Forest"
    ],
    "Accuracy (%)": [
        lr_accuracy * 100,
        svm_accuracy * 100,
        rf_accuracy * 100
    ]
})

results = results.sort_values(
    by="Accuracy (%)",
    ascending=False
)

print("MACHINE LEARNING MODEL COMPARISON")
display(results)

# Select the best model
best_model_name = results.iloc[0]["Model"]

models = {
    "Logistic Regression": lr_model,
    "Linear SVM": svm_model,
    "Random Forest": rf_model
}

best_model = models[best_model_name]

print("\nBest Model:", best_model_name)

MACHINE LEARNING MODEL COMPARISON


,Model,Accuracy (%)
1,Linear SVM,73.239437
2,Random Forest,72.635815
0,Logistic Regression,65.392354



Best Model: Linear SVM


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Create ML pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("svm", LinearSVC())
])

# Different parameters to test
parameters = {
    "tfidf__max_features": [5000, 10000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "svm__C": [0.5, 1, 2],
    "svm__class_weight": [None, "balanced"]
}

# Hyperparameter tuning
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# Train using training data only
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Test on unseen data
y_pred_tuned = best_model.predict(X_test)

# Calculate accuracy
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print("\nBEST PARAMETERS:")
print(grid_search.best_params_)

print("\nBEST CROSS-VALIDATION ACCURACY:")
print(round(grid_search.best_score_ * 100, 2), "%")

print("\nTUNED MODEL TEST ACCURACY:")
print(round(tuned_accuracy * 100, 2), "%")

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_test,
    y_pred_tuned,
    zero_division=0
))

Fitting 5 folds for each of 48 candidates, totalling 240 fits


KeyboardInterrupt: 

In [15]:
import joblib

# Restore original Linear SVM
best_model = svm_model

# Save original best model
joblib.dump(best_model, "resume_classifier.pkl")

# Save original TF-IDF vectorizer
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Best model restored and saved!")
print("Selected Model: Original Linear SVM")
print("Accuracy: 73.24%")

Best model restored and saved!
Selected Model: Original Linear SVM
Accuracy: 73.24%


In [16]:
# Technical skills dictionary

skills_dictionary = [
    "python", "java", "c", "c++", "javascript",
    "html", "css", "react", "node.js",
    "sql", "mysql", "mongodb", "oracle",
    "machine learning", "deep learning",
    "artificial intelligence", "data science",
    "data analysis", "pandas", "numpy",
    "scikit-learn", "tensorflow", "pytorch",
    "flask", "django", "fastapi",
    "aws", "azure", "google cloud",
    "docker", "kubernetes", "linux",
    "git", "github", "devops",
    "power bi", "tableau", "excel",
    "networking", "cyber security",
    "computer vision", "nlp"
]

print("Skills dictionary created successfully!")
print("Total skills:", len(skills_dictionary))

Skills dictionary created successfully!
Total skills: 42


In [17]:
def extract_skills(resume_text):

    resume_text = resume_text.lower()

    extracted_skills = []

    for skill in skills_dictionary:

        pattern = r'(?<!\w)' + re.escape(skill) + r'(?!\w)'

        if re.search(pattern, resume_text):
            extracted_skills.append(skill)

    return extracted_skills


print("Skill extraction function created successfully!")

Skill extraction function created successfully!


In [18]:
# Extract skills from every resume

df_clean["Extracted_Skills"] = df_clean["Cleaned_Resume"].apply(
    extract_skills
)

print("Skill extraction completed!")

print("\nSample resume skills:")

for i in range(5):
    print("\nResume", i + 1)
    print("Category:", df_clean["Category"].iloc[i])
    print("Skills:", df_clean["Extracted_Skills"].iloc[i])

Skill extraction completed!

Sample resume skills:

Resume 1
Category: HR
Skills: ['data analysis']

Resume 2
Category: HR
Skills: ['networking']

Resume 3
Category: HR
Skills: ['excel']

Resume 4
Category: HR
Skills: ['excel']

Resume 5
Category: HR
Skills: ['excel']


In [ ]:
from google.colab import files

uploaded_jobs = files.upload()

In [19]:
import pandas as pd

# Load Naukri job dataset
jobs_df = pd.read_csv("naukri_com-job_sample.csv")

print("Job dataset loaded successfully!")

print("\nDataset shape:")
print(jobs_df.shape)

print("\nColumn names:")
print(jobs_df.columns.tolist())

print("\nFirst 5 records:")
display(jobs_df.head())

Job dataset loaded successfully!

Dataset shape:
(22000, 14)

Column names:
['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']

First 5 records:


,company,education,experience,industry,jobdescription,jobid,joblocation_address,jobtitle,numberofpositions,payrate,postdate,site_name,skills,uniq_id
0,MM Media Pvt Ltd,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,0 - 1 yrs,Media / Entertainment / Internet,Job Description Send me Jobs like this Quali...,210516002263,Chennai,Walkin Data Entry Operator (night Shift),NaN,"1,50,000 - 2,25,000 P.A",2016-05-21 19:30:00 +0000,NaN,ITES,43b19632647068535437c774b6ca6cf8
1,find live infotech,UG: B.Tech/B.E. - Any Specialization PG:MBA/PG...,0 - 0 yrs,Advertising / PR / MR / Event Management,Job Description Send me Jobs like this Quali...,210516002391,Chennai,Work Based Onhome Based Part Time.,60.0,"1,50,000 - 2,50,000 P.A. 20000",2016-05-21 19:30:00 +0000,NaN,Marketing,d4c72325e57f89f364812b5ed5a795f0
2,Softtech Career Infosystem Pvt. Ltd,UG: Any Graduate - Any Specialization PG:Any P...,4 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this - as ...,101016900534,Bengaluru,Pl/sql Developer - SQL,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,c47df6f4cfdf5b46f1fd713ba61b9eba
3,Onboard HRServices LLP,UG: Any Graduate - Any Specialization PG:CA Do...,11 - 15 yrs,Banking / Financial Services / Broking,Job Description Send me Jobs like this - Inv...,81016900536,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Manager/ad/partner - Indirect Tax - CA,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,Accounts,115d28f140f694dd1cc61c53d03c66ae
4,Spire Technologies and Solutions Pvt. Ltd.,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,6 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this Pleas...,120916002122,Bengaluru,JAVA Technical Lead (6-8 yrs) -,4.0,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,a12553fc03bc7bcced8b1bb8963f97b4


In [20]:
print(jobs_df.columns.tolist())

['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']


In [21]:
display(jobs_df.head())

,company,education,experience,industry,jobdescription,jobid,joblocation_address,jobtitle,numberofpositions,payrate,postdate,site_name,skills,uniq_id
0,MM Media Pvt Ltd,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,0 - 1 yrs,Media / Entertainment / Internet,Job Description Send me Jobs like this Quali...,210516002263,Chennai,Walkin Data Entry Operator (night Shift),NaN,"1,50,000 - 2,25,000 P.A",2016-05-21 19:30:00 +0000,NaN,ITES,43b19632647068535437c774b6ca6cf8
1,find live infotech,UG: B.Tech/B.E. - Any Specialization PG:MBA/PG...,0 - 0 yrs,Advertising / PR / MR / Event Management,Job Description Send me Jobs like this Quali...,210516002391,Chennai,Work Based Onhome Based Part Time.,60.0,"1,50,000 - 2,50,000 P.A. 20000",2016-05-21 19:30:00 +0000,NaN,Marketing,d4c72325e57f89f364812b5ed5a795f0
2,Softtech Career Infosystem Pvt. Ltd,UG: Any Graduate - Any Specialization PG:Any P...,4 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this - as ...,101016900534,Bengaluru,Pl/sql Developer - SQL,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,c47df6f4cfdf5b46f1fd713ba61b9eba
3,Onboard HRServices LLP,UG: Any Graduate - Any Specialization PG:CA Do...,11 - 15 yrs,Banking / Financial Services / Broking,Job Description Send me Jobs like this - Inv...,81016900536,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Manager/ad/partner - Indirect Tax - CA,NaN,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,Accounts,115d28f140f694dd1cc61c53d03c66ae
4,Spire Technologies and Solutions Pvt. Ltd.,UG: B.Tech/B.E. - Any Specialization PG:Any Po...,6 - 8 yrs,IT-Software / Software Services,Job Description Send me Jobs like this Pleas...,120916002122,Bengaluru,JAVA Technical Lead (6-8 yrs) -,4.0,Not Disclosed by Recruiter,2016-10-13 16:20:55 +0000,NaN,IT Software - Application Programming,a12553fc03bc7bcced8b1bb8963f97b4


In [22]:
# Select required columns
jobs_clean = jobs_df[
    [
        "company",
        "experience",
        "jobtitle",
        "jobdescription",
        "skills",
        "joblocation_address"
    ]
].copy()

# Remove records without job title or description
jobs_clean = jobs_clean.dropna(
    subset=["jobtitle", "jobdescription"]
)

# Convert text columns into strings
text_columns = [
    "company",
    "experience",
    "jobtitle",
    "jobdescription",
    "skills",
    "joblocation_address"
]

for col in text_columns:
    jobs_clean[col] = jobs_clean[col].fillna("").astype(str).str.strip()

# Remove duplicate jobs
jobs_clean = jobs_clean.drop_duplicates(
    subset=["jobtitle", "company", "jobdescription"]
)

# Reset index
jobs_clean = jobs_clean.reset_index(drop=True)

print("Job dataset cleaning completed!")

print("Original job records:", len(jobs_df))
print("Cleaned job records:", len(jobs_clean))

display(jobs_clean.head())

Job dataset cleaning completed!
Original job records: 22000
Cleaned job records: 21848


,company,experience,jobtitle,jobdescription,skills,joblocation_address
0,MM Media Pvt Ltd,0 - 1 yrs,Walkin Data Entry Operator (night Shift),Job Description Send me Jobs like this Quali...,ITES,Chennai
1,find live infotech,0 - 0 yrs,Work Based Onhome Based Part Time.,Job Description Send me Jobs like this Quali...,Marketing,Chennai
2,Softtech Career Infosystem Pvt. Ltd,4 - 8 yrs,Pl/sql Developer - SQL,Job Description Send me Jobs like this - as ...,IT Software - Application Programming,Bengaluru
3,Onboard HRServices LLP,11 - 15 yrs,Manager/ad/partner - Indirect Tax - CA,Job Description Send me Jobs like this - Inv...,Accounts,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator..."
4,Spire Technologies and Solutions Pvt. Ltd.,6 - 8 yrs,JAVA Technical Lead (6-8 yrs) -,Job Description Send me Jobs like this Pleas...,IT Software - Application Programming,Bengaluru


In [23]:
# Combine job information into one text field

jobs_clean["Job_Combined"] = (
    jobs_clean["jobtitle"] + " " +
    jobs_clean["jobdescription"] + " " +
    jobs_clean["skills"] + " " +
    jobs_clean["company"] + " " +
    jobs_clean["experience"] + " " +
    jobs_clean["joblocation_address"]
)

# Clean combined job text
jobs_clean["Job_Combined"] = jobs_clean["Job_Combined"].apply(clean_text)

print("Job information combined successfully!")

print("\nSample combined job description:")
print(jobs_clean["Job_Combined"].iloc[0][:500])

Job information combined successfully!

Sample combined job description:
walkin data entry operator night shift job description send me jobs like this qualifications 10th to graduation any skill basic computer knowledge job requirement system or laptop type of job full time or part time languages tamil english. experience freshers experience payment details 1 form per day 5 10 form per day 50 100 form per day 500 monthly you can earn 15000 per month selection process easy selection process so what are you waiting for apply now grab best opportunity to make your carri


In [24]:
import re

def clean_text(text):

    text = str(text).lower()

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove special characters
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

print("Text cleaning function ready!")

Text cleaning function ready!


In [25]:
jobs_clean["Job_Combined"] = (
    jobs_clean["jobtitle"] + " " +
    jobs_clean["jobdescription"] + " " +
    jobs_clean["skills"] + " " +
    jobs_clean["company"] + " " +
    jobs_clean["experience"] + " " +
    jobs_clean["joblocation_address"]
)

jobs_clean["Job_Combined"] = (
    jobs_clean["Job_Combined"].apply(clean_text)
)

print("Job information combined successfully!")

print("\nSample job description:")
print(jobs_clean["Job_Combined"].iloc[0][:500])

Job information combined successfully!

Sample job description:
walkin data entry operator night shift job description send me jobs like this qualifications 10th to graduation any skill basic computer knowledge job requirement system or laptop type of job full time or part time languages tamil english. experience freshers experience payment details 1 form per day 5 10 form per day 50 100 form per day 500 monthly you can earn 15000 per month selection process easy selection process so what are you waiting for apply now grab best opportunity to make your carri


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer for job descriptions
job_tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# Convert job descriptions into numerical vectors
job_vectors = job_tfidf.fit_transform(
    jobs_clean["Job_Combined"]
)

print("Job TF-IDF feature extraction completed!")

print("Total job records:", job_vectors.shape[0])
print("Total numerical features:", job_vectors.shape[1])

Job TF-IDF feature extraction completed!
Total job records: 21848
Total numerical features: 10000


In [27]:
print("Job vector shape:", job_vectors.shape)
print("\nFirst job title:", jobs_clean["jobtitle"].iloc[0])
print("\nFirst job skills:", jobs_clean["skills"].iloc[0])

Job vector shape: (21848, 10000)

First job title: Walkin Data Entry Operator (night Shift)

First job skills: ITES


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(resume_text, top_n=5):

    # Clean the uploaded resume text
    cleaned_resume = clean_text(resume_text)

    # Convert resume into job TF-IDF feature space
    resume_vector = job_tfidf.transform([cleaned_resume])

    # Calculate similarity with all jobs
    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    ).flatten()

    # Get top matching job indexes
    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    # Prepare recommendations
    recommendations = jobs_clean.iloc[top_indices][
        [
            "jobtitle",
            "company",
            "skills",
            "experience",
            "joblocation_address"
        ]
    ].copy()

    recommendations["Match_Score"] = (
        similarity_scores[top_indices] * 100
    ).round(2)

    return recommendations.reset_index(drop=True)


print("Job recommendation engine created successfully!")

Job recommendation engine created successfully!


In [29]:
sample_resume = df_clean["Cleaned_Resume"].iloc[100]

recommendations = recommend_jobs(sample_resume, top_n=5)

print("TOP 5 RECOMMENDED JOBS")
display(recommendations)

TOP 5 RECOMMENDED JOBS


,jobtitle,company,skills,experience,joblocation_address,Match_Score
0,Payroll Admin,Sandoz Pvt. Ltd.,HR,1 - 5 yrs,Hyderabad / Secunderabad,30.60
1,HR Executive,Modern V.R. Security Force (India) Pvt. Ltd,HR,2 - 6 yrs,"Hyderabad, Bhopal, Haridwar, Gurgaon, Lucknow,...",28.51
2,Payroll Admin,Novartis Healthcare Pvt. Ltd.,HR,1 - 5 yrs,Hyderabad / Secunderabad,28.09
3,Senior Manager - HR Operations - US - Consulting,Okda Solutions,HR,8 - 13 yrs,Hyderabad,27.08
4,Payroll Executive,Global Hospitals,HR,2 - 5 yrs,Mumbai,24.39


In [30]:
print("Available variables:")

for name in ["df", "df_clean", "jobs_clean", "job_tfidf", "job_vectors"]:
    print(name, "Available:", name in globals())

Available variables:
df Available: True
df_clean Available: True
jobs_clean Available: True
job_tfidf Available: True
job_vectors Available: True


In [31]:
df_clean = df[["Resume_str", "Category"]].copy()

df_clean = df_clean.dropna(
    subset=["Resume_str", "Category"]
)

df_clean["Resume_str"] = df_clean["Resume_str"].astype(str)

df_clean["Cleaned_Resume"] = (
    df_clean["Resume_str"].apply(clean_text)
)

print("Resume dataset restored successfully!")
print("Total resumes:", len(df_clean))

Resume dataset restored successfully!
Total resumes: 2484


In [ ]:

from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [33]:

import pandas as pd
import re

df = pd.read_csv("Resume.csv")

print("Resume dataset restored!")
print("Dataset shape:", df.shape)

df_clean = df[["Resume_str", "Category"]].copy()

df_clean = df_clean.dropna(
    subset=["Resume_str", "Category"]
)

df_clean["Resume_str"] = df_clean["Resume_str"].astype(str)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_clean["Cleaned_Resume"] = (
    df_clean["Resume_str"].apply(clean_text)
)

print("Resume dataset restored successfully!")
print("Total resumes:", len(df_clean))

Resume dataset restored!
Dataset shape: (2484, 4)
Resume dataset restored successfully!
Total resumes: 2484


In [ ]:

from google.colab import files

uploaded_files = files.upload()

In [34]:
import os

print("Available CSV files:")

for file in os.listdir():
    if file.endswith(".csv"):
        print(file)

Available CSV files:
Resume.csv
naukri_com-job_sample.csv


In [35]:
import pandas as pd
import re

df = pd.read_csv("Resume.csv")

df_clean = df[["Resume_str", "Category"]].dropna().copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_clean["Cleaned_Resume"] = (
    df_clean["Resume_str"].apply(clean_text)
)

print("Resume dataset restored!")
print("Total resumes:", len(df_clean))

Resume dataset restored!
Total resumes: 2484


In [36]:
import pandas as pd
import re

# Corrected the filename to match the case-sensitive 'Resume.csv' in your workspace
df = pd.read_csv("Resume.csv")

df_clean = df[["Resume_str", "Category"]].dropna().copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_clean["Cleaned_Resume"] = (
    df_clean["Resume_str"].apply(clean_text)
)

print("Resume dataset restored successfully!")
print("Total resumes:", len(df_clean))
print(df_clean.head())

Resume dataset restored successfully!
Total resumes: 2484
                                          Resume_str Category  \
0           HR ADMINISTRATOR/MARKETING ASSOCIATE\...       HR   
1           HR SPECIALIST, US HR OPERATIONS      ...       HR   
2           HR DIRECTOR       Summary      Over 2...       HR   
3           HR SPECIALIST       Summary    Dedica...       HR   
4           HR MANAGER         Skill Highlights  ...       HR   

                                      Cleaned_Resume  
0  hr administrator marketing associate hr admini...  
1  hr specialist us hr operations summary versati...  
2  hr director summary over 20 years experience i...  
3  hr specialist summary dedicated driven and dyn...  
4  hr manager skill highlights hr skills hr depar...  


In [37]:
jobs_df = pd.read_csv("naukri_com-job_sample.csv")

jobs_clean = jobs_df[
    [
        "company",
        "experience",
        "jobtitle",
        "jobdescription",
        "skills",
        "joblocation_address"
    ]
].copy()

jobs_clean = jobs_clean.dropna(
    subset=["jobtitle", "jobdescription"]
)

for col in jobs_clean.columns:
    jobs_clean[col] = (
        jobs_clean[col].fillna("").astype(str).str.strip()
    )

jobs_clean = jobs_clean.drop_duplicates(
    subset=["jobtitle", "company", "jobdescription"]
).reset_index(drop=True)

print("Job dataset restored successfully!")
print("Total jobs:", len(jobs_clean))

Job dataset restored successfully!
Total jobs: 21848


In [38]:

from sklearn.feature_extraction.text import TfidfVectorizer

jobs_clean["Job_Combined"] = (
    jobs_clean["jobtitle"] + " " +
    jobs_clean["jobdescription"] + " " +
    jobs_clean["skills"] + " " +
    jobs_clean["company"] + " " +
    jobs_clean["experience"] + " " +
    jobs_clean["joblocation_address"]
).apply(clean_text)

job_tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

job_vectors = job_tfidf.fit_transform(
    jobs_clean["Job_Combined"]
)

print("Job TF-IDF restored successfully!")
print("Job vectors:", job_vectors.shape)

Job TF-IDF restored successfully!
Job vectors: (21848, 10000)


In [39]:

from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(resume_text, top_n=5):

    cleaned_resume = clean_text(resume_text)

    resume_vector = job_tfidf.transform([cleaned_resume])

    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    ).flatten()

    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    recommendations = jobs_clean.iloc[top_indices][
        [
            "jobtitle",
            "company",
            "skills",
            "experience",
            "joblocation_address"
        ]
    ].copy()

    recommendations["Match_Score"] = (
        similarity_scores[top_indices] * 100
    ).round(2)

    return recommendations.reset_index(drop=True)

print("Recommendation engine restored successfully!")

Recommendation engine restored successfully!


In [40]:
sample_resume = df_clean["Cleaned_Resume"].iloc[100]

recommendations = recommend_jobs(sample_resume, top_n=5)

display(recommendations)

,jobtitle,company,skills,experience,joblocation_address,Match_Score
0,Payroll Admin,Sandoz Pvt. Ltd.,HR,1 - 5 yrs,Hyderabad / Secunderabad,30.60
1,HR Executive,Modern V.R. Security Force (India) Pvt. Ltd,HR,2 - 6 yrs,"Hyderabad, Bhopal, Haridwar, Gurgaon, Lucknow,...",28.51
2,Payroll Admin,Novartis Healthcare Pvt. Ltd.,HR,1 - 5 yrs,Hyderabad / Secunderabad,28.09
3,Senior Manager - HR Operations - US - Consulting,Okda Solutions,HR,8 - 13 yrs,Hyderabad,27.08
4,Payroll Executive,Global Hospitals,HR,2 - 5 yrs,Mumbai,24.39


In [41]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 69.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 164.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [PyMuPDF]


In [42]:

import fitz
from google.colab import files

print("PDF processing libraries ready!")

PDF processing libraries ready!


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [43]:
from google.colab import files

uploaded_resume = files.upload()

Saving fresher resume sample by Voice_of_therealsai_1.pdf to fresher resume sample by Voice_of_therealsai_1.pdf


In [44]:
!pip install pymupdf
import fitz

pdf_filename = list(uploaded_resume.keys())[0]

doc = fitz.open(pdf_filename)

resume_text = ""

for page in doc:
    resume_text += page.get_text()

doc.close()

print("Resume text extracted successfully!")
print("\nExtracted Resume Content:\n")
print(resume_text[:2000])

Resume text extracted successfully!

Extracted Resume Content:

Your Name  
 xxxxx@gmail.com 
 +91999999999  
 Bengaluru, Karnataka 
 Leetcode Profile 
 HackerRank Profile 
 GitHub Profile 
 LinkedIn Profile 
EDUCATION 
 
B.Tech - Computer Science and Engineering 
Oct 20XX – May 20XX 
AITS , Rajampeta , Kadapa , A.P. 
CGPA: 9.5/10 
TECHNICAL SKILLS 
 
Programming Languages: Java, Python  
Frameworks & Technologies: Servlets, JSP, Hibernate 
Database & Web Technologies: MySQL, JDBC, HTML5, CSS3 
Developer Tools: Eclipse IDE, MySQL Workbench 
INTERNSHIPS 
 
Full Stack Web Development Intern, Jspiders Academy 
Jan 20XX – Jun 20XX 
Completion Certificate: [View Certificate 
 ] 
 Developed enterprise-level web applications utilizing Java, Servlets, JSP, JDBC, and MySQL with emphasis on 
MVC architectural patterns. 
 Implemented comprehensive CRUD operations, session management, and database design for applications. 
 Designed and integrated responsive front-end interfaces using HTML5/CSS3 w

In [45]:
from sklearn.metrics.pairwise import cosine_similarity
import re

# Ensure dependent functions and variables are loaded in this scope
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def recommend_jobs(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vector = job_tfidf.transform([cleaned_resume])
    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    ).flatten()
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    recommendations = jobs_clean.iloc[top_indices][
        [
            "jobtitle",
            "company",
            "skills",
            "experience",
            "joblocation_address"
        ]
    ].copy()
    recommendations["Match_Score"] = (
        similarity_scores[top_indices] * 100
    ).round(2)
    return recommendations.reset_index(drop=True)

# Generate recommendations from uploaded PDF
if not resume_text.strip():
    print("Error: No text found in the PDF.")
else:
    pdf_recommendations = recommend_jobs(
        resume_text,
        top_n=5
    )

    print("AI RESUME ANALYSIS")
    print("=" * 50)

    print("\nExtracted resume characters:", len(resume_text))

    print("\nTOP 5 RECOMMENDED JOBS")
    display(pdf_recommendations)

AI RESUME ANALYSIS

Extracted resume characters: 2799

TOP 5 RECOMMENDED JOBS


,jobtitle,company,skills,experience,joblocation_address,Match_Score
0,Java Web Developer,SalesBabu Business Solutions Pvt Ltd,IT Software - Application Programming,1 - 5 yrs,Bengaluru/Bangalore,27.30
1,Software Engineer,Kahuna Systems,IT Software - Application Programming,1 - 2 yrs,"Mumbai , Pune",25.49
2,Immediate Opening - Mysql Lead - Callhealth - ...,Call Health Services Pvt. Ltd.,IT Software - Application Programming,7 - 10 yrs,Hyderabad,25.37
3,Java/j2ee Developer - Servlets/jsp/jdbc,Career Infosystems,IT Software - Application Programming,6 - 11 yrs,Bengaluru/Bangalore,24.40
4,Java Developer,Atulsia Technologies,IT Software - Application Programming,2 - 7 yrs,Mumbai,24.26


In [ ]:
from google.colab import files

uploaded_files = files.upload()

In [46]:
import os

print("Files available:")
print(os.listdir())

Files available:
['.config', 'fresher resume sample by Voice_of_therealsai_1.pdf', 'tfidf_vectorizer.pkl', 'drive', 'resume_classifier.pkl', 'Resume.csv', 'resume_model.pkl', 'naukri_com-job_sample.csv', 'sample_data']


In [48]:
from google.colab import files

uploaded_second = files.upload()

Saving resume updated 1 (1).pdf to resume updated 1 (1).pdf


In [49]:
import os

print("Resume dataset:", os.path.exists("Resume.CSV"))
print("Job dataset:", os.path.exists("naukri_com-job_sample.csv"))

Resume dataset: False
Job dataset: True


In [50]:
import pandas as pd
import re

df = pd.read_csv("Resume.csv")

df_clean = df[["Resume_str", "Category"]].dropna().copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#.\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_clean["Cleaned_Resume"] = (
    df_clean["Resume_str"].apply(clean_text)
)

print("Resume dataset restored!")
print("Total resumes:", len(df_clean))

Resume dataset restored!
Total resumes: 2484


In [51]:
import os

print("Resume dataset:", os.path.exists("Resume.csv"))
print("Job dataset:", os.path.exists("naukri_com-job_sample.csv"))

Resume dataset: True
Job dataset: True


In [52]:
import pandas as pd

jobs_df = pd.read_csv("naukri_com-job_sample.csv")

jobs_clean = jobs_df[
    [
        "company",
        "experience",
        "jobtitle",
        "jobdescription",
        "skills",
        "joblocation_address"
    ]
].copy()

jobs_clean = jobs_clean.dropna(
    subset=["jobtitle", "jobdescription"]
)

for col in jobs_clean.columns:
    jobs_clean[col] = (
        jobs_clean[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

jobs_clean = jobs_clean.drop_duplicates(
    subset=["jobtitle", "company", "jobdescription"]
).reset_index(drop=True)

print("Job dataset restored successfully!")
print("Clean job records:", len(jobs_clean))

Job dataset restored successfully!
Clean job records: 21848


In [53]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

jobs_clean["Job_Combined"] = (
    jobs_clean["jobtitle"] + " " +
    jobs_clean["jobdescription"] + " " +
    jobs_clean["skills"] + " " +
    jobs_clean["company"] + " " +
    jobs_clean["experience"] + " " +
    jobs_clean["joblocation_address"]
).apply(clean_text)

job_tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

job_vectors = job_tfidf.fit_transform(
    jobs_clean["Job_Combined"]
)

def recommend_jobs(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vector = job_tfidf.transform([cleaned_resume])

    scores = cosine_similarity(
        resume_vector, job_vectors
    ).flatten()

    top_indices = scores.argsort()[-top_n:][::-1]

    result = jobs_clean.iloc[top_indices][
        ["jobtitle", "company", "skills",
         "experience", "joblocation_address"]
    ].copy()

    result["Match_Score"] = (
        scores[top_indices] * 100
    ).round(2)

    return result.reset_index(drop=True)

print("Recommendation engine restored!")
print("Job vectors:", job_vectors.shape)

Recommendation engine restored!
Job vectors: (21848, 10000)


In [54]:

# Check whether PDF text is available
if "resume_text" not in globals() or not resume_text.strip():
    print("PDF text is missing. Please upload and extract your PDF again.")
else:
    pdf_recommendations = recommend_jobs(
        resume_text,
        top_n=5
    )

    print("AI RESUME ANALYSIS COMPLETED")
    print("=" * 45)

    print("Resume characters:", len(resume_text))

    print("\nTOP 5 RECOMMENDED JOBS")
    display(pdf_recommendations)

AI RESUME ANALYSIS COMPLETED
Resume characters: 2799

TOP 5 RECOMMENDED JOBS


,jobtitle,company,skills,experience,joblocation_address,Match_Score
0,Java Web Developer,SalesBabu Business Solutions Pvt Ltd,IT Software - Application Programming,1 - 5 yrs,Bengaluru/Bangalore,27.30
1,Software Engineer,Kahuna Systems,IT Software - Application Programming,1 - 2 yrs,"Mumbai , Pune",25.49
2,Immediate Opening - Mysql Lead - Callhealth - ...,Call Health Services Pvt. Ltd.,IT Software - Application Programming,7 - 10 yrs,Hyderabad,25.37
3,Java/j2ee Developer - Servlets/jsp/jdbc,Career Infosystems,IT Software - Application Programming,6 - 11 yrs,Bengaluru/Bangalore,24.40
4,Java Developer,Atulsia Technologies,IT Software - Application Programming,2 - 7 yrs,Mumbai,24.26


In [ ]:

from google.colab import files

uploaded_resume = files.upload()

In [55]:
!pip install PyMuPDF -q

import fitz

# Get uploaded PDF filename automatically
pdf_filename = [
    name for name in uploaded_resume.keys()
    if name.lower().endswith(".pdf")
][0]

# Open PDF
doc = fitz.open(pdf_filename)

# Extract text
resume_text = ""

for page in doc:
    resume_text += page.get_text() + "\n"

doc.close()

print("PDF TEXT EXTRACTION SUCCESSFUL!")
print("Total characters:", len(resume_text))

print("\nEXTRACTED RESUME CONTENT:")
print(resume_text[:2000])

PDF TEXT EXTRACTION SUCCESSFUL!
Total characters: 2800

EXTRACTED RESUME CONTENT:
Your Name  
 xxxxx@gmail.com 
 +91999999999  
 Bengaluru, Karnataka 
 Leetcode Profile 
 HackerRank Profile 
 GitHub Profile 
 LinkedIn Profile 
EDUCATION 
 
B.Tech - Computer Science and Engineering 
Oct 20XX – May 20XX 
AITS , Rajampeta , Kadapa , A.P. 
CGPA: 9.5/10 
TECHNICAL SKILLS 
 
Programming Languages: Java, Python  
Frameworks & Technologies: Servlets, JSP, Hibernate 
Database & Web Technologies: MySQL, JDBC, HTML5, CSS3 
Developer Tools: Eclipse IDE, MySQL Workbench 
INTERNSHIPS 
 
Full Stack Web Development Intern, Jspiders Academy 
Jan 20XX – Jun 20XX 
Completion Certificate: [View Certificate 
 ] 
 Developed enterprise-level web applications utilizing Java, Servlets, JSP, JDBC, and MySQL with emphasis on 
MVC architectural patterns. 
 Implemented comprehensive CRUD operations, session management, and database design for applications. 
 Designed and integrated responsive front-end interfaces 

In [56]:
# Check required components
print("Resume text available:", bool(resume_text.strip()))
print("Recommendation function available:", "recommend_jobs" in globals())
print("Job dataset available:", "jobs_clean" in globals())

# Generate recommendations
pdf_recommendations = recommend_jobs(
    resume_text,
    top_n=5
)

print("\nAI RESUME ANALYSIS COMPLETED!")
print("=" * 45)

print("\nTOP 5 RECOMMENDED JOBS:")
display(pdf_recommendations)

Resume text available: True
Recommendation function available: True
Job dataset available: True

AI RESUME ANALYSIS COMPLETED!

TOP 5 RECOMMENDED JOBS:


,jobtitle,company,skills,experience,joblocation_address,Match_Score
0,Java Web Developer,SalesBabu Business Solutions Pvt Ltd,IT Software - Application Programming,1 - 5 yrs,Bengaluru/Bangalore,27.30
1,Software Engineer,Kahuna Systems,IT Software - Application Programming,1 - 2 yrs,"Mumbai , Pune",25.49
2,Immediate Opening - Mysql Lead - Callhealth - ...,Call Health Services Pvt. Ltd.,IT Software - Application Programming,7 - 10 yrs,Hyderabad,25.37
3,Java/j2ee Developer - Servlets/jsp/jdbc,Career Infosystems,IT Software - Application Programming,6 - 11 yrs,Bengaluru/Bangalore,24.40
4,Java Developer,Atulsia Technologies,IT Software - Application Programming,2 - 7 yrs,Mumbai,24.26


In [57]:
import re
import pandas as pd

# Common technical skills
skill_list = [
    "python", "java", "c++", "c", "sql", "mysql",
    "html", "css", "javascript", "react", "node.js",
    "machine learning", "deep learning", "artificial intelligence",
    "data science", "data analysis", "pandas", "numpy",
    "scikit-learn", "tensorflow", "pytorch",
    "aws", "azure", "docker", "kubernetes",
    "git", "github", "flask", "django",
    "excel", "power bi", "tableau",
    "communication", "leadership"
]

# Find skills in uploaded resume
resume_lower = resume_text.lower()

resume_skills = [
    skill for skill in skill_list
    if re.search(r'(?<!\w)' + re.escape(skill) + r'(?!\w)', resume_lower)
]

print("SKILLS FOUND IN YOUR RESUME:")
print(resume_skills)

# Compare with recommended jobs
skill_results = []

for _, job in pdf_recommendations.iterrows():

    job_skills_text = str(job["skills"]).lower()

    required_skills = [
        skill for skill in skill_list
        if re.search(
            r'(?<!\w)' + re.escape(skill) + r'(?!\w)',
            job_skills_text
        )
    ]

    missing_skills = [
        skill for skill in required_skills
        if skill not in resume_skills
    ]

    skill_results.append({
        "Job Title": job["jobtitle"],
        "Available Skills": ", ".join(resume_skills),
        "Required Skills": ", ".join(required_skills),
        "Missing Skills": ", ".join(missing_skills) if missing_skills else "None"
    })

skill_analysis = pd.DataFrame(skill_results)

print("\nMISSING SKILLS ANALYSIS")
display(skill_analysis)

SKILLS FOUND IN YOUR RESUME:
['python', 'java', 'mysql', 'github']

MISSING SKILLS ANALYSIS


,Job Title,Available Skills,Required Skills,Missing Skills
0,Java Web Developer,"python, java, mysql, github",,None
1,Software Engineer,"python, java, mysql, github",,None
2,Immediate Opening - Mysql Lead - Callhealth - ...,"python, java, mysql, github",,None
3,Java/j2ee Developer - Servlets/jsp/jdbc,"python, java, mysql, github",,None
4,Java Developer,"python, java, mysql, github",,None


In [58]:
import re
import pandas as pd

skill_list = [
    "python", "java", "c++", "sql", "mysql",
    "html", "css", "javascript", "react",
    "machine learning", "deep learning",
    "artificial intelligence", "data science",
    "data analysis", "pandas", "numpy",
    "scikit-learn", "tensorflow", "pytorch",
    "aws", "azure", "docker", "kubernetes",
    "git", "github", "flask", "django",
    "excel", "power bi", "tableau",
    "communication", "leadership"
]

def find_skills(text):
    text = str(text).lower()

    found = []

    for skill in skill_list:
        pattern = r'(?<!\w)' + re.escape(skill) + r'(?!\w)'

        if re.search(pattern, text):
            found.append(skill)

    return found

# Extract resume skills
resume_skills = find_skills(resume_text)

results = []

for _, job in pdf_recommendations.iterrows():

    job_text = (
        str(job["jobtitle"]) + " " +
        str(job["skills"]) + " " +
        str(job["experience"])
    )

    required = find_skills(job_text)

    missing = [
        skill for skill in required
        if skill not in resume_skills
    ]

    results.append({
        "Job Title": job["jobtitle"],
        "Resume Skills": ", ".join(resume_skills),
        "Required Skills": ", ".join(required),
        "Missing Skills": ", ".join(missing) if missing else "None"
    })

skill_analysis = pd.DataFrame(results)

print("IMPROVED SKILL GAP ANALYSIS")
display(skill_analysis)

IMPROVED SKILL GAP ANALYSIS


,Job Title,Resume Skills,Required Skills,Missing Skills
0,Java Web Developer,"python, java, mysql, github",java,None
1,Software Engineer,"python, java, mysql, github",,None
2,Immediate Opening - Mysql Lead - Callhealth - ...,"python, java, mysql, github",mysql,None
3,Java/j2ee Developer - Servlets/jsp/jdbc,"python, java, mysql, github",java,None
4,Java Developer,"python, java, mysql, github",java,None


In [59]:
!pip install gradio -q

import gradio as gr
import fitz

print("Gradio installed successfully!")

Gradio installed successfully!


In [60]:


import gradio as gr
import fitz
import pandas as pd
import re

# Skill detection function
def detect_skills(text):
    skill_list = [
        "python", "java", "c++", "sql", "mysql",
        "html", "css", "javascript", "react",
        "machine learning", "deep learning",
        "artificial intelligence", "data science",
        "data analysis", "pandas", "numpy",
        "scikit-learn", "tensorflow", "pytorch",
        "aws", "azure", "docker", "kubernetes",
        "git", "github", "flask", "django",
        "excel", "power bi", "tableau"
    ]

    text = str(text).lower()

    return [
        skill for skill in skill_list
        if re.search(
            r'(?<!\w)' + re.escape(skill) + r'(?!\w)',
            text
        )
    ]


# Main resume analysis function
def analyze_resume(pdf_file):

    if pdf_file is None:
        return "Please upload a PDF resume.", pd.DataFrame(), ""

    # Extract PDF text
    doc = fitz.open(pdf_file)
    resume = ""

    for page in doc:
        resume += page.get_text() + "\n"

    doc.close()

    if not resume.strip():
        return "No readable text found in PDF.", pd.DataFrame(), ""

    # Generate job recommendations
    recommendations = recommend_jobs(resume, top_n=5)

    # Detect resume skills
    resume_skills = detect_skills(resume)

    # Missing skills analysis
    skill_results = []

    for _, job in recommendations.iterrows():

        job_text = (
            str(job["jobtitle"]) + " " +
            str(job["skills"]) + " " +
            str(job["experience"])
        )

        required = detect_skills(job_text)

        missing = [
            skill for skill in required
            if skill not in resume_skills
        ]

        skill_results.append({
            "Job Title": job["jobtitle"],
            "Required Skills": ", ".join(required),
            "Missing Skills": ", ".join(missing) if missing else "None"
        })

    skill_table = pd.DataFrame(skill_results)

    summary = (
        "## Resume Analysis Completed!\n\n"
        f"**Resume characters:** {len(resume)}\n\n"
        f"**Skills detected:** {', '.join(resume_skills) if resume_skills else 'None'}"
    )

    return summary, recommendations, skill_table


# Create web interface
with gr.Blocks(title="AI Resume Analyzer") as app:

    gr.Markdown(
        "# AI Resume Analyzer and Job Recommendation System\n"
        "### Upload your resume to discover matching jobs and skill gaps."
    )

    pdf_input = gr.File(
        label="Upload Your Resume (PDF)",
        file_types=[".pdf"],
        type="filepath"
    )

    analyze_button = gr.Button("Analyze My Resume")

    summary_output = gr.Markdown()

    gr.Markdown("## Top 5 Recommended Jobs")

    jobs_output = gr.Dataframe(
        label="Job Recommendations",
        interactive=False
    )

    gr.Markdown("## Missing Skills Analysis")

    skills_output = gr.Dataframe(
        label="Skill Gap Analysis",
        interactive=False
    )

    analyze_button.click(
        fn=analyze_resume,
        inputs=pdf_input,
        outputs=[
            summary_output,
            jobs_output,
            skills_output
        ]
    )

print("AI Resume Analyzer application created!")

AI Resume Analyzer application created!


In [61]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4ccf1fc974a5fe076.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [62]:

import re
import pandas as pd

def analyze_resume_sections(resume_text):

    text = resume_text.lower()

    sections = {
        "Contact Information": [
            "email", "phone", "mobile", "contact"
        ],
        "Professional Summary": [
            "summary", "professional summary",
            "career objective", "objective"
        ],
        "Education": [
            "education", "academic qualification",
            "qualification"
        ],
        "Technical Skills": [
            "skills", "technical skills",
            "programming skills"
        ],
        "Projects": [
            "projects", "academic projects",
            "personal projects"
        ],
        "Work Experience": [
            "experience", "work experience",
            "employment"
        ],
        "Certifications": [
            "certifications", "certificates"
        ]
    }

    results = []

    for section, keywords in sections.items():

        found = any(
            keyword in text for keyword in keywords
        )

        results.append({
            "Resume Section": section,
            "Status": "Present" if found else "Missing",
            "Recommendation": (
                "Good" if found
                else "Consider adding this section"
            )
        })

    return pd.DataFrame(results)


# Analyze uploaded resume
section_analysis = analyze_resume_sections(resume_text)

print("RESUME SECTION ANALYSIS")
display(section_analysis)

RESUME SECTION ANALYSIS


,Resume Section,Status,Recommendation
0,Contact Information,Missing,Consider adding this section
1,Professional Summary,Missing,Consider adding this section
2,Education,Present,Good
3,Technical Skills,Present,Good
4,Projects,Present,Good
5,Work Experience,Missing,Consider adding this section
6,Certifications,Present,Good


In [63]:
import re
import pandas as pd

def analyze_resume_content(resume_text):

    text = resume_text.lower()

    checks = {
        "Measurable Achievements": [
            "%", "increased", "reduced", "improved",
            "achieved", "saved"
        ],
        "Action Words": [
            "developed", "created", "designed",
            "implemented", "managed", "built",
            "analyzed", "programmed"
        ],
        "Project Details": [
            "project", "developed", "technologies",
            "tools", "using", "application"
        ],
        "Technical Experience": [
            "python", "java", "sql", "machine learning",
            "html", "css", "database"
        ]
    }

    results = []

    for item, keywords in checks.items():

        found = [word for word in keywords if word in text]

        results.append({
            "Resume Quality Check": item,
            "Status": "Good" if found else "Needs Improvement",
            "Detected Keywords": ", ".join(found) if found else "None",
            "Suggestion": (
                "Add more relevant details"
                if not found else
                "Try adding specific examples and results"
            )
        })

    return pd.DataFrame(results)


content_analysis = analyze_resume_content(resume_text)

print("RESUME CONTENT ANALYSIS")
display(content_analysis)

RESUME CONTENT ANALYSIS


,Resume Quality Check,Status,Detected Keywords,Suggestion
0,Measurable Achievements,Needs Improvement,None,Add more relevant details
1,Action Words,Good,"developed, created, designed, implemented",Try adding specific examples and results
2,Project Details,Good,"project, developed, technologies, tools, using...",Try adding specific examples and results
3,Technical Experience,Good,"python, java, sql, html, css, database",Try adding specific examples and results


In [64]:
def generate_resume_suggestions(resume_text):

    text = resume_text.lower()
    suggestions = []

    # Check important resume sections
    if not any(word in text for word in
               ["summary", "professional summary", "objective"]):
        suggestions.append(
            "Add a professional summary describing your career goals."
        )

    if "education" not in text:
        suggestions.append(
            "Add an Education section with your degree and college details."
        )

    if not any(word in text for word in ["project", "projects"]):
        suggestions.append(
            "Include academic or personal projects with their technologies."
        )

    if not any(word in text for word in
               ["certification", "certifications", "certificate"]):
        suggestions.append(
            "Add relevant certifications to strengthen your profile."
        )

    # Check achievements
    if not any(word in text for word in
               ["%", "increased", "reduced", "improved", "achieved"]):
        suggestions.append(
            "Include measurable achievements, such as percentages or results."
        )

    # Check action words
    action_words = [
        "developed", "created", "designed",
        "implemented", "built", "analyzed"
    ]

    if not any(word in text for word in action_words):
        suggestions.append(
            "Use strong action words such as Developed, Designed and Implemented."
        )

    # Display suggestions
    if not suggestions:
        suggestions.append(
            "Your resume includes several important elements. "
            "Review it for clarity, accuracy and measurable results."
        )

    return suggestions


# Generate suggestions for uploaded resume
resume_suggestions = generate_resume_suggestions(resume_text)

print("PERSONALIZED RESUME IMPROVEMENT SUGGESTIONS")
print("-" * 50)

for i, suggestion in enumerate(resume_suggestions, 1):
    print(f"{i}. {suggestion}")

PERSONALIZED RESUME IMPROVEMENT SUGGESTIONS
--------------------------------------------------
1. Add a professional summary describing your career goals.
2. Include measurable achievements, such as percentages or results.


In [65]:
def calculate_resume_score(resume_text):

    text = resume_text.lower()

    score = 0
    details = []

    # 1. Resume sections (40 marks)
    sections = {
        "Professional Summary": ["summary", "objective"],
        "Education": ["education"],
        "Technical Skills": ["skills", "technical skills"],
        "Projects": ["project", "projects"],
        "Experience": ["experience", "employment"],
        "Certifications": ["certification", "certificate"]
    }

    section_score = 0

    for section, keywords in sections.items():
        if any(word in text for word in keywords):
            section_score += 1
            details.append([section, "Present", "Included"])
        else:
            details.append([section, "Missing", "Consider adding it"])

    score += (section_score / len(sections)) * 40

    # 2. Technical skills (20 marks)
    technical_skills = [
        "python", "java", "sql", "machine learning",
        "html", "css", "javascript", "aws",
        "pandas", "numpy"
    ]

    skill_count = sum(skill in text for skill in technical_skills)
    score += min(skill_count / 5, 1) * 20

    # 3. Projects (15 marks)
    if "project" in text or "projects" in text:
        score += 15

    # 4. Certifications (10 marks)
    if "certification" in text or "certificate" in text:
        score += 10

    # 5. Measurable achievements (10 marks)
    achievement_words = [
        "%", "increased", "reduced", "improved", "achieved"
    ]

    if any(word in text for word in achievement_words):
        score += 10

    # 6. Action words (5 marks)
    action_words = [
        "developed", "created", "designed",
        "implemented", "built", "analyzed"
    ]

    if any(word in text for word in action_words):
        score += 5

    final_score = round(min(score, 100), 2)

    print("YOUR RESUME SCORE")
    print("=" * 35)
    print(f"Resume Score: {final_score}/100")

    if final_score >= 80:
        print("Rating: Excellent")
    elif final_score >= 60:
        print("Rating: Good")
    elif final_score >= 40:
        print("Rating: Average")
    else:
        print("Rating: Needs Improvement")

    print("\nSECTION ANALYSIS")
    display(pd.DataFrame(
        details,
        columns=["Resume Section", "Status", "Suggestion"]
    ))

    return final_score


# Calculate score
resume_score = calculate_resume_score(resume_text)

YOUR RESUME SCORE
Resume Score: 76.67/100
Rating: Good

SECTION ANALYSIS


,Resume Section,Status,Suggestion
0,Professional Summary,Missing,Consider adding it
1,Education,Present,Included
2,Technical Skills,Present,Included
3,Projects,Present,Included
4,Experience,Missing,Consider adding it
5,Certifications,Present,Included


In [82]:

import pandas as pd

def calculate_resume_score(resume_text):

    text = str(resume_text).lower()

    score = 0
    details = []

    # 1. Resume sections (40 marks)
    sections = {
        "Professional Summary": ["summary", "objective"],
        "Education": ["education"],
        "Technical Skills": ["skills", "technical skills"],
        "Projects": ["project", "projects"],
        "Experience": ["experience", "employment"],
        "Certifications": ["certification", "certificate"]
    }

    section_score = 0

    for section, keywords in sections.items():
        present = any(word in text for word in keywords)

        if present:
            section_score += 1
            details.append([section, "Present", "Included"])
        else:
            details.append([section, "Missing", "Consider adding it"])

    section_marks = (section_score / len(sections)) * 40
    score += section_marks

    # 2. Technical skills (20 marks)
    technical_skills = [
        "python", "java", "sql", "machine learning",
        "html", "css", "javascript", "aws",
        "pandas", "numpy"
    ]

    skill_count = sum(skill in text for skill in technical_skills)
    skill_marks = min(skill_count / 5, 1) * 20
    score += skill_marks

    # 3. Projects (15 marks)
    project_marks = 15 if any(
        word in text for word in ["project", "projects"]
    ) else 0
    score += project_marks

    # 4. Certifications (10 marks)
    certification_marks = 10 if any(
        word in text for word in ["certification", "certificate"]
    ) else 0
    score += certification_marks

    # 5. Measurable achievements (10 marks)
    achievement_words = [
        "%", "increased", "reduced", "improved", "achieved"
    ]

    achievement_marks = 10 if any(
        word in text for word in achievement_words
    ) else 0
    score += achievement_marks

    # 6. Action words (5 marks)
    action_words = [
        "developed", "created", "designed",
        "implemented", "built", "analyzed"
    ]

    action_marks = 5 if any(
        word in text for word in action_words
    ) else 0
    score += action_marks

    final_score = round(min(score, 100), 2)

    print("YOUR RESUME SCORE")
    print("=" * 35)
    print(f"Resume Score: {final_score}/100")

    if final_score >= 80:
        print("Rating: Excellent")
    elif final_score >= 60:
        print("Rating: Good")
    elif final_score >= 40:
        print("Rating: Average")
    else:
        print("Rating: Needs Improvement")

    print("\nSECTION ANALYSIS")

    display(pd.DataFrame(
        details,
        columns=["Resume Section", "Status", "Suggestion"]
    ))

    print("\nSCORE BREAKDOWN")
    display(pd.DataFrame({
        "Criteria": [
            "Resume Sections", "Technical Skills",
            "Projects", "Certifications",
            "Achievements", "Action Words"
        ],
        "Marks Obtained": [
            round(section_marks, 2),
            round(skill_marks, 2),
            project_marks,
            certification_marks,
            achievement_marks,
            action_marks
        ],
        "Maximum Marks": [40, 20, 15, 10, 10, 5]
    }))

    return final_score


# Calculate resume score
resume_score = calculate_resume_score(resume_text)

YOUR RESUME SCORE
Resume Score: 76.67/100
Rating: Good

SECTION ANALYSIS


,Resume Section,Status,Suggestion
0,Professional Summary,Missing,Consider adding it
1,Education,Present,Included
2,Technical Skills,Present,Included
3,Projects,Present,Included
4,Experience,Missing,Consider adding it
5,Certifications,Present,Included



SCORE BREAKDOWN


,Criteria,Marks Obtained,Maximum Marks
0,Resume Sections,26.67,40
1,Technical Skills,20.00,20
2,Projects,15.00,15
3,Certifications,10.00,10
4,Achievements,0.00,10
5,Action Words,5.00,5


In [66]:
import gradio as gr
import fitz
import pandas as pd

def analyze_resume(pdf_file):

    if pdf_file is None:
        return (
            "Please upload a PDF resume.",
            0,
            pd.DataFrame(),
            pd.DataFrame(),
            "No suggestions available.",
            pd.DataFrame(),
            pd.DataFrame()
        )

    # Extract text from PDF
    doc = fitz.open(pdf_file)
    resume_text = ""

    for page in doc:
        resume_text += page.get_text()

    doc.close()

    if not resume_text.strip():
        return (
            "Could not extract text from PDF.",
            0,
            pd.DataFrame(),
            pd.DataFrame(),
            "Please upload a text-based PDF.",
            pd.DataFrame(),
            pd.DataFrame()
        )

    # Resume score
    score = calculate_resume_score(resume_text)

    # Section analysis
    sections = analyze_resume_sections(resume_text)

    # Content analysis
    content = analyze_resume_content(resume_text)

    # Improvement suggestions
    suggestions = generate_resume_suggestions(resume_text)

    # Job recommendations
    jobs = recommend_jobs(resume_text, top_n=5)

    # Skill gap analysis
    skill_result = detect_skills(resume_text)

    suggestion_text = "\n".join(
        [f"{i}. {s}" for i, s in enumerate(suggestions, 1)]
    )

    return (
        "Resume successfully analyzed!",
        score,
        sections,
        content,
        suggestion_text,
        jobs,
        skill_result
    )


with gr.Blocks() as app:

    gr.Markdown("# AI Resume Analyzer and Job Recommendation System")
    gr.Markdown("Upload your resume PDF to analyze your profile.")

    pdf_input = gr.File(
        label="Upload Resume PDF",
        type="filepath",
        file_types=[".pdf"]
    )

    analyze_button = gr.Button("Analyze Resume")

    status_output = gr.Textbox(label="Analysis Status")
    score_output = gr.Number(label="Resume Score / 100")

    section_output = gr.Dataframe(label="Resume Section Analysis")
    content_output = gr.Dataframe(label="Resume Content Analysis")

    suggestion_output = gr.Textbox(
        label="Personalized Improvement Suggestions",
        lines=8
    )

    job_output = gr.Dataframe(label="Recommended Jobs")
    skill_output = gr.Dataframe(label="Skill Gap Analysis")

    analyze_button.click(
        fn=analyze_resume,
        inputs=pdf_input,
        outputs=[
            status_output,
            score_output,
            section_output,
            content_output,
            suggestion_output,
            job_output,
            skill_output
        ]
    )


app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://52334c403ea26dfc2f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [67]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.8 MB/s  0:00:00


In [68]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch
import os

def generate_resume_report():

    filename = "/content/Resume_Analysis_Report.pdf"

    doc = SimpleDocTemplate(filename, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph(
        "AI Resume Analyzer - Analysis Report",
        styles["Title"]
    ))
    elements.append(Spacer(1, 20))

    elements.append(Paragraph(
        f"Resume Score: {resume_score}/100",
        styles["Heading2"]
    ))
    elements.append(Spacer(1, 15))

    # Section analysis
    elements.append(Paragraph(
        "Resume Section Analysis", styles["Heading2"]
    ))

    section_data = [section_analysis.columns.tolist()]
    section_data += section_analysis.values.tolist()

    elements.append(Table(section_data))
    elements.append(Spacer(1, 15))

    # Content analysis
    elements.append(Paragraph(
        "Resume Content Analysis", styles["Heading2"]
    ))

    content_data = [content_analysis.columns.tolist()]
    content_data += content_analysis.values.tolist()

    elements.append(Table(content_data))
    elements.append(Spacer(1, 15))

    # Suggestions
    elements.append(Paragraph(
        "Improvement Suggestions", styles["Heading2"]
    ))

    for i, suggestion in enumerate(resume_suggestions, 1):
        elements.append(Paragraph(
            f"{i}. {suggestion}", styles["BodyText"]
        ))
        elements.append(Spacer(1, 6))

    doc.build(elements)

    return filename


report_file = generate_resume_report()

print("PDF report generated successfully!")
print(report_file)

PDF report generated successfully!
/content/Resume_Analysis_Report.pdf


In [69]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def generate_job_report():

    filename = "/content/Job_Recommendation_Report.pdf"

    doc = SimpleDocTemplate(filename, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph(
        "AI Job Recommendation Report",
        styles["Title"]
    ))

    elements.append(Spacer(1, 20))

    elements.append(Paragraph(
        "Top 5 Recommended Jobs",
        styles["Heading2"]
    ))

    # Use the existing jobs_clean and recommendation function
    recommended_jobs = recommend_jobs(resume_text, top_n=5)

    columns = [
        col for col in [
            "company", "jobtitle",
            "joblocation_address", "Match_Score"
        ]
        if col in recommended_jobs.columns
    ]

    if not columns:
        raise ValueError("Expected job recommendation columns were not found.")

    report_data = [columns]

    for _, job in recommended_jobs[columns].iterrows():
        report_data.append([
            str(value)[:150] for value in job.tolist()
        ])

    table = Table(report_data, repeatRows=1)

    table.setStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ])

    elements.append(table)

    doc.build(elements)

    return filename


job_report_file = generate_job_report()

print("Job recommendation report generated!")
print(job_report_file)

Job recommendation report generated!
/content/Job_Recommendation_Report.pdf


In [70]:
from google.colab import files

files.download("/content/Job_Recommendation_Report.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [71]:
import pandas as pd

job_role_skills = {
    "Python Developer": [
        "Python", "OOP", "SQL", "Git",
        "Django", "Flask", "REST API"
    ],

    "Data Analyst": [
        "Python", "SQL", "Excel", "Power BI",
        "Tableau", "Pandas", "Statistics"
    ],

    "Machine Learning Engineer": [
        "Python", "NumPy", "Pandas",
        "Scikit-learn", "TensorFlow",
        "Machine Learning", "Deep Learning"
    ],

    "Web Developer": [
        "HTML", "CSS", "JavaScript",
        "React", "Node.js", "SQL", "Git"
    ],

    "Java Developer": [
        "Java", "OOP", "Spring Boot",
        "SQL", "Hibernate", "Git"
    ],

    "Cloud Engineer": [
        "AWS", "Linux", "Docker",
        "Kubernetes", "Networking", "Python"
    ]
}


def recommend_skills(resume_text, selected_role):

    text = resume_text.lower()

    required_skills = job_role_skills[selected_role]

    results = []

    for skill in required_skills:

        if skill.lower() in text:
            status = "Already Present"
            suggestion = "Continue improving this skill"
        else:
            status = "Missing"
            suggestion = "Consider learning this skill"

        results.append({
            "Skill": skill,
            "Status": status,
            "Recommendation": suggestion
        })

    return pd.DataFrame(results)


print("Job-role-based skill recommendation system created!")

Job-role-based skill recommendation system created!


In [72]:
selected_role = "Python Developer"

role_skill_analysis = recommend_skills(
    resume_text,
    selected_role
)

print("Selected Job Role:", selected_role)
display(role_skill_analysis)

Selected Job Role: Python Developer


,Skill,Status,Recommendation
0,Python,Already Present,Continue improving this skill
1,OOP,Missing,Consider learning this skill
2,SQL,Already Present,Continue improving this skill
3,Git,Already Present,Continue improving this skill
4,Django,Missing,Consider learning this skill
5,Flask,Missing,Consider learning this skill
6,REST API,Missing,Consider learning this skill


In [73]:
import gradio as gr
import fitz
import pandas as pd

def analyze_resume_with_role(pdf_file, selected_role):

    if pdf_file is None:
        return "Please upload a PDF resume.", pd.DataFrame()

    # Extract resume text
    doc = fitz.open(pdf_file)
    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    if not text.strip():
        return "Could not extract PDF text.", pd.DataFrame()

    # Analyze skills for selected job role
    role_analysis = recommend_skills(text, selected_role)

    return (
        f"Resume analyzed for: {selected_role}",
        role_analysis
    )


with gr.Blocks() as role_app:

    gr.Markdown("# Job Role-Based Skill Recommendation")

    pdf_input = gr.File(
        label="Upload Resume PDF",
        type="filepath",
        file_types=[".pdf"]
    )

    role_input = gr.Dropdown(
        choices=list(job_role_skills.keys()),
        label="Select Target Job Role",
        value="Python Developer"
    )

    analyze_button = gr.Button("Analyze Skills")

    status_output = gr.Textbox(label="Analysis Status")

    skill_output = gr.Dataframe(
        label="Required Skills and Skill Gaps"
    )

    analyze_button.click(
        fn=analyze_resume_with_role,
        inputs=[pdf_input, role_input],
        outputs=[status_output, skill_output]
    )


role_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46034c7e8ffbee2605.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [74]:
import gradio as gr
import fitz
import pandas as pd

def complete_resume_analysis(pdf_file, selected_role):

    if pdf_file is None:
        raise gr.Error("Please upload a PDF resume.")

    # Extract resume text
    doc = fitz.open(pdf_file)
    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    if not text.strip():
        raise gr.Error("No readable text found in the PDF.")

    # Resume score
    score = calculate_resume_score(text)

    # Resume section analysis
    sections = analyze_resume_sections(text)

    # Resume content analysis
    content = analyze_resume_content(text)

    # Improvement suggestions
    suggestions = generate_resume_suggestions(text)

    # Job recommendations
    jobs = recommend_jobs(text, top_n=5)

    # Existing skill-gap analysis
    skill_gaps = detect_skills(text)

    # Target role skill analysis
    role_skills = recommend_skills(text, selected_role)

    suggestion_text = "\n".join(
        [f"{i}. {s}" for i, s in enumerate(suggestions, 1)]
    )

    return (
        score,
        sections,
        content,
        suggestion_text,
        jobs,
        skill_gaps,
        role_skills
    )


with gr.Blocks(title="AI Resume Analyzer") as final_app:

    gr.Markdown(
        "# AI Resume Analyzer and Job Recommendation System"
    )

    gr.Markdown(
        "Upload your resume to analyze your profile "
        "and discover suitable job opportunities."
    )

    pdf_input = gr.File(
        label="Upload Resume PDF",
        type="filepath",
        file_types=[".pdf"]
    )

    role_input = gr.Dropdown(
        choices=list(job_role_skills.keys()),
        value="Python Developer",
        label="Select Your Target Job Role"
    )

    analyze_button = gr.Button(
        "Analyze My Resume",
        variant="primary"
    )

    score_output = gr.Number(
        label="Resume Score / 100"
    )

    with gr.Tab("Resume Section Analysis"):
        section_output = gr.Dataframe()

    with gr.Tab("Content Analysis"):
        content_output = gr.Dataframe()

    with gr.Tab("Improvement Suggestions"):
        suggestion_output = gr.Textbox(lines=10)

    with gr.Tab("Recommended Jobs"):
        jobs_output = gr.Dataframe()

    with gr.Tab("Skill Gap Analysis"):
        skill_output = gr.Dataframe()

    with gr.Tab("Target Role Skills"):
        role_skill_output = gr.Dataframe()

    analyze_button.click(
        fn=complete_resume_analysis,
        inputs=[pdf_input, role_input],
        outputs=[
            score_output,
            section_output,
            content_output,
            suggestion_output,
            jobs_output,
            skill_output,
            role_skill_output
        ]
    )


final_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d249c8d4123141b55a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [75]:
!pip install -q reportlab

import os
import html
import fitz
import pandas as pd

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)

def create_complete_report(pdf_file, selected_role):

    if pdf_file is None:
        raise ValueError("Please upload a resume PDF.")

    # Extract resume text
    doc = fitz.open(pdf_file)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()

    if not text.strip():
        raise ValueError("Could not extract readable text.")

    # Run all analysis functions
    score = calculate_resume_score(text)
    sections = analyze_resume_sections(text)
    content = analyze_resume_content(text)
    suggestions = generate_resume_suggestions(text)
    jobs = recommend_jobs(text, top_n=5)
    skill_gaps = detect_skills(text)
    role_skills = recommend_skills(text, selected_role)

    filename = "/content/Complete_Resume_Analysis_Report.pdf"

    styles = getSampleStyleSheet()
    elements = []

    def add_heading(title):
        elements.append(Paragraph(title, styles["Heading2"]))
        elements.append(Spacer(1, 8))

    def add_dataframe(df):
        if not isinstance(df, pd.DataFrame) or df.empty:
            elements.append(Paragraph("No data available.", styles["BodyText"]))
            return

        data = [[
            Paragraph(html.escape(str(c)), styles["BodyText"])
            for c in df.columns
        ]]

        for _, row in df.iterrows():
            data.append([
                Paragraph(
                    html.escape(str(value))[:300],
                    styles["BodyText"]
                )
                for value in row
            ])

        table = Table(data, repeatRows=1)
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("FONTSIZE", (0, 0), (-1, -1), 8),
            ("PADDING", (0, 0), (-1, -1), 5)
        ]))

        elements.append(table)
        elements.append(Spacer(1, 15))

    # Report title
    elements.append(Paragraph(
        "AI Resume Analyzer - Complete Report",
        styles["Title"]
    ))
    elements.append(Spacer(1, 15))

    elements.append(Paragraph(
        f"Resume Score: {score}/100",
        styles["Heading1"]
    ))

    elements.append(Paragraph(
        f"Selected Target Role: {html.escape(selected_role)}",
        styles["Normal"]
    ))
    elements.append(Spacer(1, 20))

    add_heading("1. Resume Section Analysis")
    add_dataframe(sections)

    add_heading("2. Resume Content Analysis")
    add_dataframe(content)

    add_heading("3. Improvement Suggestions")

    for i, suggestion in enumerate(suggestions, 1):
        elements.append(Paragraph(
            f"{i}. {html.escape(suggestion)}",
            styles["BodyText"]
        ))
        elements.append(Spacer(1, 6))

    elements.append(Spacer(1, 12))

    add_heading("4. Recommended Jobs")
    add_dataframe(jobs)

    add_heading("5. General Skill Gap Analysis")
    add_dataframe(skill_gaps)

    add_heading("6. Target Role Skill Analysis")
    add_dataframe(role_skills)

    elements.append(Spacer(1, 15))
    elements.append(Paragraph(
        "Note: Resume score and skill checks are rule-based. "
        "Job match scores represent text similarity, not hiring probability. "
        "Job listings come from the supplied dataset and may not be current.",
        styles["Italic"]
    ))

    report = SimpleDocTemplate(filename, pagesize=A4)
    report.build(elements)

    return filename


print("Complete PDF report generator created successfully!")

Complete PDF report generator created successfully!


In [76]:
report_path = create_complete_report(
    "/content/fresher resume sample by Voice_of_therealsai_1.pdf",
    "Python Developer"
)

print("Report created:", report_path)

YOUR RESUME SCORE
Resume Score: 76.67/100
Rating: Good

SECTION ANALYSIS


,Resume Section,Status,Suggestion
0,Professional Summary,Missing,Consider adding it
1,Education,Present,Included
2,Technical Skills,Present,Included
3,Projects,Present,Included
4,Experience,Missing,Consider adding it
5,Certifications,Present,Included


Report created: /content/Complete_Resume_Analysis_Report.pdf


In [77]:
import gradio as gr

with gr.Blocks(title="AI Resume Analyzer") as final_app:

    gr.Markdown("# AI Resume Analyzer and Job Recommendation System")
    gr.Markdown("Upload your resume and analyze your career opportunities.")

    pdf_input = gr.File(
        label="Upload Resume PDF",
        type="filepath",
        file_types=[".pdf"]
    )

    role_input = gr.Dropdown(
        choices=list(job_role_skills.keys()),
        value="Python Developer",
        label="Select Target Job Role"
    )

    analyze_button = gr.Button("Analyze My Resume")

    score_output = gr.Number(label="Resume Score / 100")

    with gr.Tab("Section Analysis"):
        section_output = gr.Dataframe()

    with gr.Tab("Content Analysis"):
        content_output = gr.Dataframe()

    with gr.Tab("Improvement Suggestions"):
        suggestion_output = gr.Textbox(lines=8)

    with gr.Tab("Recommended Jobs"):
        jobs_output = gr.Dataframe()

    with gr.Tab("Skill Gap Analysis"):
        skill_output = gr.Dataframe()

    with gr.Tab("Target Role Skills"):
        role_skill_output = gr.Dataframe()

    analyze_button.click(
        fn=complete_resume_analysis,
        inputs=[pdf_input, role_input],
        outputs=[
            score_output,
            section_output,
            content_output,
            suggestion_output,
            jobs_output,
            skill_output,
            role_skill_output
        ]
    )

    gr.Markdown("## Download Your Complete Analysis Report")

    report_button = gr.Button("Generate PDF Report")

    report_output = gr.File(label="Download Your Report")

    report_button.click(
        fn=create_complete_report,
        inputs=[pdf_input, role_input],
        outputs=report_output
    )


final_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f367c834c07ecc604c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [78]:
import gradio as gr

custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
}

#main-title {
    text-align: center;
}

#analyze-btn {
    background: #2563eb;
    color: white;
    font-weight: bold;
}

#report-btn {
    background: #15803d;
    color: white;
    font-weight: bold;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title="AI Resume Analyzer"
) as professional_app:

    gr.Markdown(
        """
        # AI Resume Analyzer & Job Recommendation System
        ### Intelligent Resume Analysis | Career Guidance | Job Matching
        """,
        elem_id="main-title"
    )

    with gr.Row():
        with gr.Column(scale=2):
            pdf_input = gr.File(
                label="Upload Your Resume (PDF)",
                type="filepath",
                file_types=[".pdf"]
            )

        with gr.Column(scale=2):
            role_input = gr.Dropdown(
                choices=list(job_role_skills.keys()),
                value="Python Developer",
                label="Select Your Target Job Role"
            )

    analyze_button = gr.Button(
        "Analyze My Resume",
        elem_id="analyze-btn"
    )

    score_output = gr.Number(
        label="Resume Score (out of 100)"
    )

    with gr.Tabs():

        with gr.Tab("Resume Sections"):
            section_output = gr.Dataframe(
                label="Resume Section Analysis"
            )

        with gr.Tab("Content Analysis"):
            content_output = gr.Dataframe(
                label="Resume Content Analysis"
            )

        with gr.Tab("Improvement Suggestions"):
            suggestion_output = gr.Textbox(
                label="Personalized Suggestions",
                lines=10
            )

        with gr.Tab("Job Recommendations"):
            jobs_output = gr.Dataframe(
                label="Top Recommended Jobs"
            )

        with gr.Tab("Skill Gap Analysis"):
            skill_output = gr.Dataframe(
                label="Detected Skills"
            )

        with gr.Tab("Target Role Skills"):
            role_skill_output = gr.Dataframe(
                label="Skills Required for Selected Role"
            )

    analyze_button.click(
        fn=complete_resume_analysis,
        inputs=[pdf_input, role_input],
        outputs=[
            score_output,
            section_output,
            content_output,
            suggestion_output,
            jobs_output,
            skill_output,
            role_skill_output
        ]
    )

    gr.Markdown("---")
    gr.Markdown("## Download Your Complete Analysis Report")

    report_button = gr.Button(
        "Generate PDF Report",
        elem_id="report-btn"
    )

    report_output = gr.File(
        label="Your Downloadable Report"
    )

    report_button.click(
        fn=create_complete_report,
        inputs=[pdf_input, role_input],
        outputs=report_output
    )

    gr.Markdown(
        """
        ---
        **Project:** AI Resume Analyzer and Job Recommendation System

        **Technology:** Python | Machine Learning | NLP | TF-IDF |
        Scikit-learn | Gradio
        """
    )


professional_app.launch(share=True)

/tmp/ipykernel_431/1035085100.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a96e6ccaedba6c561.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [79]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_resume_keywords(resume_text, top_n=15):

    # Split resume into sentences
    sentences = re.split(r'[.!?\n]+', resume_text)

    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.split()) >= 3
    ]

    if not sentences:
        return pd.DataFrame(columns=["Keyword", "Importance Score"])

    # Apply TF-IDF
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=100
    )

    try:
        matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return pd.DataFrame(columns=["Keyword", "Importance Score"])

    # Calculate average TF-IDF score
    scores = matrix.mean(axis=0).A1
    keywords = vectorizer.get_feature_names_out()

    results = pd.DataFrame({
        "Keyword": keywords,
        "Importance Score": scores
    })

    results = results.sort_values(
        by="Importance Score",
        ascending=False
    ).head(top_n)

    results["Importance Score"] = results[
        "Importance Score"
    ].round(4)

    return results.reset_index(drop=True)


print("NLP keyword extraction function created successfully!")

NLP keyword extraction function created successfully!


In [80]:
resume_keywords = extract_resume_keywords(
    resume_text,
    top_n=15
)

print("IMPORTANT RESUME KEYWORDS")
display(resume_keywords)

IMPORTANT RESUME KEYWORDS


,Keyword,Importance Score
0,java,0.0546
1,mysql,0.0524
2,20xx,0.0520
3,web,0.0518
4,stack,0.0471
5,comprehensive,0.0438
6,tech,0.0436
7,database,0.0403
8,portal,0.0394
9,stack web,0.0380


In [81]:
import gradio as gr

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Resume Analyzer"
) as professional_app:

    gr.Markdown("""
    # AI Resume Analyzer and Job Recommendation System
    ### Resume Analysis | NLP | Job Matching | Career Guidance
    """)

    with gr.Row():
        pdf_input = gr.File(
            label="Upload Resume PDF",
            type="filepath",
            file_types=[".pdf"]
        )

        role_input = gr.Dropdown(
            choices=list(job_role_skills.keys()),
            value="Python Developer",
            label="Select Target Job Role"
        )

    analyze_button = gr.Button(
        "Analyze My Resume",
        variant="primary"
    )

    score_output = gr.Number(label="Resume Score / 100")

    with gr.Tabs():

        with gr.Tab("Resume Sections"):
            section_output = gr.Dataframe()

        with gr.Tab("Content Analysis"):
            content_output = gr.Dataframe()

        with gr.Tab("Improvement Suggestions"):
            suggestion_output = gr.Textbox(lines=10)

        with gr.Tab("Recommended Jobs"):
            jobs_output = gr.Dataframe()

        with gr.Tab("Skill Gap Analysis"):
            skill_output = gr.Dataframe()

        with gr.Tab("Target Role Skills"):
            role_skill_output = gr.Dataframe()

        with gr.Tab("NLP Keywords"):
            keyword_output = gr.Dataframe(
                label="Important Resume Keywords"
            )

    def complete_analysis_with_keywords(pdf_file, selected_role):

        results = complete_resume_analysis(
            pdf_file, selected_role
        )

        if pdf_file is None:
            raise gr.Error("Please upload a PDF resume.")

        doc = fitz.open(pdf_file)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()

        keywords = extract_resume_keywords(text, top_n=15)

        return (*results, keywords)

    analyze_button.click(
        fn=complete_analysis_with_keywords,
        inputs=[pdf_input, role_input],
        outputs=[
            score_output,
            section_output,
            content_output,
            suggestion_output,
            jobs_output,
            skill_output,
            role_skill_output,
            keyword_output
        ]
    )

    gr.Markdown("## Download Complete Resume Report")

    report_button = gr.Button(
        "Generate PDF Report",
        variant="secondary"
    )

    report_output = gr.File(label="Download Report")

    report_button.click(
        fn=create_complete_report,
        inputs=[pdf_input, role_input],
        outputs=report_output
    )

    gr.Markdown("""
    ---
    **Technology:** Python | Machine Learning | NLP |
    TF-IDF | Scikit-learn | Gradio
    """)

professional_app.launch(share=True)

/tmp/ipykernel_431/448848589.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ce52b3288ba0955744.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
